# Retry Yearly Descriptions

This notebook fixes `topic_yearly_descriptions.csv` files in two ways:

1. **Update stale labels** — Rows that still have `Topic_X` as their label get the corrected label from `topic_labels.csv`
2. **Regenerate descriptions** — For those same rows, the yearly description is regenerated with the correct label (since the old description references `Topic_X` in its text)

Also checks for any other yearly description failures (empty, missing) and retries with increasing temperature.

**Temperature schedule:** 0.3 → 0.5 → 0.7 → 1.0

In [1]:
import os
import re
import json
import time
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from datetime import datetime
from itertools import groupby
from collections import defaultdict
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

## Configuration

In [2]:
LIST_MODELS = ["lda", "dtm", "bertopic", "top2vec", "topicGpt"]
LIST_SUBJECT = ["cs", "math", "physics"]

BASE_DIR = Path("../../results")
LOG_DIR = Path("../../models/labeling/retry_logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

# LLM Configuration
LLM_API_URL = "http://localhost:1234/v1/chat/completions"
LLM_MODEL = "mistralai/ministral-3-3b"
LLM_MAX_TOKENS = 4096

# Retry temperature schedule
TEMPERATURE_SCHEDULE = [0.3, 0.5, 0.7, 1.0]

print(f"Models: {LIST_MODELS}")
print(f"Subjects: {LIST_SUBJECT}")
print(f"Temperature schedule: {TEMPERATURE_SCHEDULE}")
print(f"Error logs: {LOG_DIR}")

Models: ['lda', 'dtm', 'bertopic', 'top2vec', 'topicGpt']
Subjects: ['cs', 'math', 'physics']
Temperature schedule: [0.3, 0.5, 0.7, 1.0]
Error logs: ../../models/labeling/retry_logs


## LLM & Parsing Helpers

In [3]:
def call_llm(system_prompt: str, user_prompt: str, temperature: float, max_retries: int = 3) -> str:
    """Call LM Studio API with specific temperature and retry logic."""
    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": LLM_MAX_TOKENS,
    }
    
    for attempt in range(max_retries):
        try:
            resp = requests.post(
                LLM_API_URL,
                headers={"Content-Type": "application/json"},
                json=payload,
                timeout=120
            )
            resp.raise_for_status()
            data = resp.json()
            
            if "choices" in data:
                return data["choices"][0]["message"]["content"].strip()
            elif "content" in data:
                return data["content"].strip()
            elif "output" in data:
                return data["output"].strip()
            else:
                return str(data)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"    Network retry {attempt+1}/{max_retries} after {wait}s: {e}")
                time.sleep(wait)
            else:
                return f"[LLM_ERROR] {e}"

def clean_and_parse_json(response: str) -> tuple:
    """Parse JSON from LLM response. Returns (parsed_dict, error_msg)."""
    if not response or response.startswith("[LLM_ERROR]"):
        return None, f"LLM returned error: {response}"
    
    text = re.sub(r"```json\s*|```", "", response).strip()
    
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1:
        return None, f"No JSON braces found. Raw: {response[:300]}"
    
    json_str = text[start:end+1]
    json_str = json_str.replace('\n', ' ').replace('\r', '')
    
    try:
        return json.loads(json_str), None
    except json.JSONDecodeError as e:
        error1 = str(e)
    
    # Fallback: regex extraction
    try:
        match = re.search(r'"yearly_description":\s*"(.*?)"', json_str, re.DOTALL)
        if match:
            return {"yearly_description": match.group(1).strip()}, None
    except:
        pass
    
    return None, f"JSON parse failed: {error1}. Raw: {json_str[:300]}"

# Test LLM connection
test = call_llm("You are helpful.", "Say OK.", temperature=0.3)
print(f"LLM test: {test[:80]}")

LLM test: Understood! How can I assist you today? 😊


## Scan for Failures

Two types of failures:
1. **Stale label** — `label` column matches `Topic_X` pattern (needs regeneration with correct label)
2. **Missing description** — `yearly_description` is empty, NaN, or "No description available."

In [4]:
all_failures = []  # List of dicts with model, subject, topic_id, year, failure_type

for model in LIST_MODELS:
    for subject in LIST_SUBJECT:
        yearly_path = BASE_DIR / model / "temporal" / subject / "topic_yearly_descriptions.csv"
        labels_path = BASE_DIR / model / "temporal" / subject / "topic_labels.csv"
        
        if not yearly_path.exists() or not labels_path.exists():
            print(f"  [SKIP] {model}/{subject}: files not found")
            continue
        
        yearly_df = pd.read_csv(yearly_path)
        labels_df = pd.read_csv(labels_path)
        
        # Build corrected label map from topic_labels.csv
        label_map = dict(zip(labels_df["topic_id"], labels_df["label"]))
        
        stale_count = 0
        missing_count = 0
        
        for idx, row in yearly_df.iterrows():
            failure_types = []
            
            # Check stale label
            label = str(row.get("label", ""))
            if re.match(r'^Topic_\d+$', label):
                corrected = label_map.get(row["topic_id"], label)
                if corrected != label:  # Only if we have a real fix
                    failure_types.append("stale_label")
                    stale_count += 1
            
            # Check missing/empty description
            desc = str(row.get("yearly_description", ""))
            if desc == "No description available." or desc == "nan" or desc.strip() == "":
                failure_types.append("missing_description")
                missing_count += 1
            
            if failure_types:
                all_failures.append({
                    "model": model,
                    "subject": subject,
                    "topic_id": row["topic_id"],
                    "year": row["year"],
                    "old_label": label,
                    "corrected_label": label_map.get(row["topic_id"], label),
                    "old_description": str(row.get("yearly_description", ""))[:80],
                    "failure_types": failure_types
                })
        
        total = len(yearly_df)
        if stale_count > 0 or missing_count > 0:
            print(f"  {model}/{subject}: {total} rows — stale_label={stale_count}, missing_desc={missing_count}")
        else:
            print(f"  {model}/{subject}: {total} rows — OK")

print(f"\nTOTAL ROWS TO FIX: {len(all_failures)}")
if all_failures:
    failures_df = pd.DataFrame(all_failures)
    print("\nBreakdown:")
    print(failures_df.groupby(["model", "subject"]).size().to_string())

  lda/cs: 1266 rows — OK
  lda/math: 1289 rows — OK
  lda/physics: 1287 rows — OK
  dtm/cs: 1300 rows — OK
  dtm/math: 1300 rows — OK
  dtm/physics: 1300 rows — OK
  bertopic/cs: 4328 rows — OK
  bertopic/math: 3572 rows — OK
  bertopic/physics: 300 rows — OK
  top2vec/cs: 5921 rows — OK
  top2vec/math: 4868 rows — OK
  top2vec/physics: 5058 rows — OK
  topicGpt/cs: 2221 rows — OK
  topicGpt/math: 1551 rows — OK
  topicGpt/physics: 987 rows — OK

TOTAL ROWS TO FIX: 0


## Prompt for Yearly Descriptions (same as original)

In [5]:
YEARLY_SYSTEM_PROMPT = """You are an expert academic topic analyst.
Given a topic label and the representative keywords from a specific year,
write a simple 1-2 sentence description of what this topic focused on in that year.

OUTPUT RULES:
1. Return ONLY valid JSON: {"yearly_description": "..."}
2. The description should be 1-2 sentences, plain and concise.
3. Use PLAIN TEXT only. No markdown, no bolding (**), and no bullet points (-).
4. If you use quotes inside values, use 'single quotes'.
5. Keep the description on ONE SINGLE LINE."""

YEARLY_USER_TEMPLATE = """Topic Label: {label}
Subject Area: {subject}
Year: {year}

Keywords for this topic in {year}:
{words}

Write a simple 1-2 sentence description of what this topic focused on in {year}.
Return ONLY valid JSON: {{"yearly_description": "..."}}"""

print("Prompts loaded.")

Prompts loaded.


## Retry Engine

In [6]:
def load_topic_words_for_year(model: str, subject: str, topic_id: int, year: int) -> str:
    """Load top words for a specific topic and year."""
    path = BASE_DIR / model / "temporal" / subject / "topic_word_evolution.csv"
    df = pd.read_csv(path)
    row = df[(df["topic_id"] == topic_id) & (df["year"] == year)]
    if row.empty:
        return ""
    return str(row.iloc[0]["top_words"]).strip()

def retry_yearly_description(model: str, subject: str, topic_id: int, year: int,
                              label: str, failure_types: list) -> dict:
    """
    Retry a single yearly description with escalating temperature.
    Returns dict with full results and error log.
    """
    words = load_topic_words_for_year(model, subject, topic_id, year)
    if not words:
        return {
            "success": False,
            "yearly_description": None,
            "resolved_at_temp": None,
            "attempts": [{"temperature": 0, "raw_response": "", 
                         "parse_error": "No words found for this topic/year", "success": False}]
        }
    
    user_prompt = YEARLY_USER_TEMPLATE.format(
        label=label,
        subject=subject,
        year=year,
        words=words
    )
    
    attempt_logs = []
    
    for temp in TEMPERATURE_SCHEDULE:
        raw_response = call_llm(YEARLY_SYSTEM_PROMPT, user_prompt, temperature=temp)
        parsed, error_msg = clean_and_parse_json(raw_response)
        
        attempt_log = {
            "temperature": temp,
            "raw_response": raw_response[:500],
            "parse_error": error_msg,
            "success": False
        }
        
        if parsed:
            desc = parsed.get("yearly_description", "")
            if desc and desc != "No description available." and len(desc) > 10:
                attempt_log["success"] = True
                attempt_logs.append(attempt_log)
                return {
                    "success": True,
                    "yearly_description": desc,
                    "resolved_at_temp": temp,
                    "attempts": attempt_logs
                }
            else:
                attempt_log["parse_error"] = f"Parsed but description invalid (len={len(desc) if desc else 0})"
        
        attempt_logs.append(attempt_log)
    
    return {
        "success": False,
        "yearly_description": None,
        "resolved_at_temp": None,
        "attempts": attempt_logs
    }

print("Retry engine ready.")

Retry engine ready.


## Run Retries

For each failed row:
- Use the **corrected label** from `topic_labels.csv`
- Regenerate the yearly description via LLM with temperature escalation
- Log all attempts for human review

In [7]:
retry_results = []           # Summary for logging
full_retry_data = {}         # (model, subject, topic_id, year) -> {label, yearly_description}
error_logs = []              # Detailed per-attempt logs
success_count = 0
fail_count = 0

sorted_failures = sorted(all_failures, key=lambda x: (x["model"], x["subject"]))

for (model, subject), group in groupby(sorted_failures, key=lambda x: (x["model"], x["subject"])):
    failures_list = list(group)
    print(f"\n{'='*60}")
    print(f"RETRYING: {model.upper()} / {subject.upper()} ({len(failures_list)} rows)")
    print(f"{'='*60}")
    
    for failure in tqdm(failures_list, desc=f"Retry {model}/{subject}"):
        topic_id = failure["topic_id"]
        year = failure["year"]
        corrected_label = failure["corrected_label"]
        
        result = retry_yearly_description(
            model, subject, topic_id, year,
            corrected_label, failure["failure_types"]
        )
        
        # Store full data for CSV update
        if result["success"]:
            full_retry_data[(model, subject, topic_id, year)] = {
                "label": corrected_label,
                "yearly_description": result["yearly_description"]
            }
        
        retry_entry = {
            "model": model,
            "subject": subject,
            "topic_id": topic_id,
            "year": year,
            "old_label": failure["old_label"],
            "corrected_label": corrected_label,
            "failure_types": ", ".join(failure["failure_types"]),
            "retry_success": result["success"],
            "new_description_preview": str(result["yearly_description"])[:150] if result["yearly_description"] else None,
            "resolved_at_temp": result["resolved_at_temp"]
        }
        retry_results.append(retry_entry)
        
        if result["success"]:
            success_count += 1
        else:
            fail_count += 1
            print(f"  FAIL Topic {topic_id}/{year} STILL FAILED")
        
        # Log all attempts
        for attempt in result["attempts"]:
            error_logs.append({
                "model": model,
                "subject": subject,
                "topic_id": topic_id,
                "year": year,
                "temperature": attempt["temperature"],
                "success": attempt["success"],
                "parse_error": attempt["parse_error"],
                "raw_response_preview": attempt["raw_response"][:300]
            })

print(f"\n{'='*60}")
print(f"RETRY COMPLETE: {success_count} fixed, {fail_count} still failed out of {len(all_failures)} total")
print(f"{'='*60}")


RETRY COMPLETE: 0 fixed, 0 still failed out of 0 total


## Update CSV Files

In [8]:
# Group updates by file for efficient batch writes
updates_by_file = defaultdict(list)
for (model, subject, topic_id, year), data in full_retry_data.items():
    updates_by_file[(model, subject)].append((topic_id, year, data))

total_updated = 0
for (model, subject), updates in sorted(updates_by_file.items()):
    yearly_path = BASE_DIR / model / "temporal" / subject / "topic_yearly_descriptions.csv"
    df = pd.read_csv(yearly_path)
    
    for topic_id, year, data in updates:
        mask = (df["topic_id"] == topic_id) & (df["year"] == year)
        if mask.any():
            df.loc[mask, "label"] = data["label"]
            df.loc[mask, "yearly_description"] = data["yearly_description"]
    
    df.to_csv(yearly_path, index=False)
    total_updated += len(updates)
    print(f"  {model}/{subject}: updated {len(updates)} rows")

# Also update labels for rows that weren't regenerated but have stale Topic_X labels
# (in case some LLM retries failed but the label itself is still fixable)
print("\nUpdating remaining stale labels (no regeneration needed)...")
label_only_fixes = 0
for model in LIST_MODELS:
    for subject in LIST_SUBJECT:
        yearly_path = BASE_DIR / model / "temporal" / subject / "topic_yearly_descriptions.csv"
        labels_path = BASE_DIR / model / "temporal" / subject / "topic_labels.csv"
        if not yearly_path.exists() or not labels_path.exists():
            continue
        
        yearly_df = pd.read_csv(yearly_path)
        labels_df = pd.read_csv(labels_path)
        label_map = dict(zip(labels_df["topic_id"], labels_df["label"]))
        
        changes = 0
        for idx, row in yearly_df.iterrows():
            current_label = str(row.get("label", ""))
            if re.match(r'^Topic_\d+$', current_label):
                corrected = label_map.get(row["topic_id"], current_label)
                if corrected != current_label:
                    yearly_df.at[idx, "label"] = corrected
                    changes += 1
        
        if changes > 0:
            yearly_df.to_csv(yearly_path, index=False)
            label_only_fixes += changes
            print(f"  {model}/{subject}: fixed {changes} stale labels")

print(f"\nTotal: {total_updated} rows regenerated, {label_only_fixes} additional labels fixed.")


Updating remaining stale labels (no regeneration needed)...

Total: 0 rows regenerated, 0 additional labels fixed.


## Save Error Logs

In [9]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# 1. Full error logs
if error_logs:
    error_df = pd.DataFrame(error_logs)
    error_log_path = LOG_DIR / f"yearly_retry_error_log_{timestamp}.csv"
    error_df.to_csv(error_log_path, index=False)
    print(f"Error log: {error_log_path} ({len(error_df)} entries)")

# 2. Summary of retry results
if retry_results:
    results_df = pd.DataFrame(retry_results)
    results_path = LOG_DIR / f"yearly_retry_results_{timestamp}.csv"
    results_df.to_csv(results_path, index=False)
    print(f"Results: {results_path} ({len(results_df)} entries)")

    # 3. Still-failed entries
    still_failed = results_df[results_df["retry_success"] == False]
    if len(still_failed) > 0:
        still_failed_path = LOG_DIR / f"yearly_still_failed_{timestamp}.csv"
        still_failed.to_csv(still_failed_path, index=False)
        print(f"\nStill-failed: {still_failed_path} ({len(still_failed)} entries)")
        print("\nBreakdown:")
        print(still_failed.groupby(["model", "subject"]).size().to_string())
    else:
        print("\nAll failures were resolved!")
else:
    print("No retries were needed.")

No retries were needed.


## Summary Report

In [10]:
print("=" * 60)
print("YEARLY DESCRIPTIONS RETRY SUMMARY")
print("=" * 60)
print(f"\nTotal rows to fix: {len(all_failures)}")
print(f"Successfully regenerated: {success_count} ({100*success_count/max(len(all_failures),1):.1f}%)")
print(f"Still failed: {fail_count}")

if retry_results:
    results_df = pd.DataFrame(retry_results)
    successful = results_df[results_df['retry_success']]
    if len(successful) > 0:
        print(f"\nTemperature distribution:")
        temp_dist = successful['resolved_at_temp'].value_counts().sort_index()
        for temp, count in temp_dist.items():
            print(f"  temp={temp}: {count} descriptions fixed")
    
    print(f"\nPer-model breakdown:")
    for model in LIST_MODELS:
        model_results = results_df[results_df['model'] == model]
        if len(model_results) == 0:
            continue
        fixed = model_results['retry_success'].sum()
        total = len(model_results)
        print(f"  {model}: {fixed}/{total} fixed")

YEARLY DESCRIPTIONS RETRY SUMMARY

Total rows to fix: 0
Successfully regenerated: 0 (0.0%)
Still failed: 0


## Verify Updated Files

In [11]:
print("Post-retry verification:")
print("=" * 60)

total_stale = 0
total_missing = 0

for model in LIST_MODELS:
    for subject in LIST_SUBJECT:
        yearly_path = BASE_DIR / model / "temporal" / subject / "topic_yearly_descriptions.csv"
        labels_path = BASE_DIR / model / "temporal" / subject / "topic_labels.csv"
        if not yearly_path.exists():
            continue
        
        yearly_df = pd.read_csv(yearly_path)
        labels_df = pd.read_csv(labels_path)
        label_map = dict(zip(labels_df["topic_id"], labels_df["label"]))
        
        stale = 0
        missing = 0
        for _, row in yearly_df.iterrows():
            label = str(row.get("label", ""))
            desc = str(row.get("yearly_description", ""))
            if re.match(r'^Topic_\d+$', label):
                corrected = label_map.get(row["topic_id"], label)
                if corrected != label:
                    stale += 1
            if desc == "No description available." or desc == "nan" or desc.strip() == "":
                missing += 1
        
        total = len(yearly_df)
        issues = []
        if stale > 0: issues.append(f"{stale} stale labels")
        if missing > 0: issues.append(f"{missing} missing desc")
        status = ", ".join(issues) if issues else "all good"
        print(f"  {model}/{subject}: {total} rows - {status}")
        total_stale += stale
        total_missing += missing

print(f"\nTotal remaining: {total_stale} stale labels, {total_missing} missing descriptions")

Post-retry verification:
  lda/cs: 1266 rows - all good
  lda/math: 1289 rows - all good
  lda/physics: 1287 rows - all good
  dtm/cs: 1300 rows - all good
  dtm/math: 1300 rows - all good
  dtm/physics: 1300 rows - all good
  bertopic/cs: 4328 rows - all good
  bertopic/math: 3572 rows - all good
  bertopic/physics: 300 rows - all good
  top2vec/cs: 5921 rows - all good
  top2vec/math: 4868 rows - all good
  top2vec/physics: 5058 rows - all good
  topicGpt/cs: 2221 rows - all good
  topicGpt/math: 1551 rows - all good
  topicGpt/physics: 987 rows - all good

Total remaining: 0 stale labels, 0 missing descriptions
